# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MuhammadFaizan0023/FlyRank_ML_internship_repo/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

Finding#1: <br>
Captured Traffic Value<br>
The defensible proxy here is `clicks × CPC`, not `impressions × CPC`<br>
This is a value proxy, not booked revenue. CPC is a cost-per-click benchmark, so the defensible portfolio read is
captured click-equivalent value from clicks × CPC. Using impressions × CPC inflates the number dramatically
because impressions are not billable clicks.<br>
1. INTENT: informational, <br>
PAGES: 36.8K, <br>
CLICKS: 486.9K, <br>
CPC: 0.15,<br>VALUE: $74.9K

2. INTENT:commercial, <br>
PAGES:9.9K,<br> CLICKS:127.3K,<br> CPC:0.68, <br>VALUE:$86.1K

3. INTENT: transactional,<br> PAGES: 10.5K,<br> CLICKS:121.7K,<br> CPC:0.76,<br> VALUE: $92.0K

4. INTENT:navigational,<br> PAGES:108,<br> CLICKS:1.6K,<br> CPC:0.35,<br> VALUE:$549



Finding #2<br>
Click capture by position tier<br>
Weighted portfolio CTR declines sharply as visibility moves away from the top of the results<br>
Weighted CTR by position tier<br>
1. Top 3: 0.423%
2. Page 1 (4-10): 0.339%
3. Striking: 0.325%
4. Page 3-5: 0.163%
5. Deep: 0.050%

These are portfolio-level weighted CTRs computed from total clicks divided by total impressions in each position tier.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import os, getpass
import duckdb

In [2]:
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':                f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':                f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':                 f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample':          f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':             f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

for name, src in TABLES.items():
    n = con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0]
    print(f'{name:22} {n:>12,} rows')

dim_clients                     104 rows
dim_content                 519,606 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_daily               78,835,655 rows
fact_daily_sample        11,694,072 rows
fact_query_90d            2,414,248 rows


In [24]:
import pandas as pd

# --- Finding #1: Captured Traffic Value Audit ---
# Label: VALUE = CLICKS * CPC
# We join dim_content (intent) with fact_daily_sample (performance metrics)
print("--- Auditing Finding #1: Intent-Based Traffic Value ---")
audit_f1 = con.sql(f"""
    SELECT
        c.main_intent,
        COUNT(DISTINCT c.content_hash_id) as PAGES,
        SUM(f.gsc_clicks) as CLICKS,
        -- Using the benchmark CPCs provided in the finding to calculate Value
        CASE
            WHEN c.main_intent = 'informational' THEN 0.15
            WHEN c.main_intent = 'commercial' THEN 0.68
            WHEN c.main_intent = 'transactional' THEN 0.76
            WHEN c.main_intent = 'navigational' THEN 0.35
            ELSE 0.0
        END as CPC
    FROM {TABLES['dim_content']} c
    JOIN {TABLES['fact_daily']} f ON c.content_hash_id = f.content_hash_id
    GROUP BY 1, 4
""").df()

audit_f1['VALUE'] = audit_f1['CLICKS'] * audit_f1['CPC']
display(audit_f1.sort_values('VALUE', ascending=False))

# --- Finding #2: Click Capture by Position Tier Audit ---
# Label: Weighted CTR = Sum(Clicks) / Sum(Impressions)
print("\n--- Auditing Finding #2: Weighted CTR by Position Tier ---")
def bucket_tier(pos):
    if pos <= 3: return '1. Top 3'
    if pos <= 10: return '2. Page 1 (4-10)'
    if pos <= 20: return '3. Striking'
    if pos <= 50: return '4. Page 3-5'
    return '5. Deep'

# Query fact_query_90d for the 90d window claim
df_q = con.sql(f"SELECT avg_position_90d, clicks_90d, impressions_90d FROM {TABLES['fact_query_90d']}").df()
df_q['tier'] = df_q['avg_position_90d'].apply(bucket_tier)

audit_f2 = df_q.groupby('tier').agg({
    'clicks_90d': 'sum',
    'impressions_90d': 'sum'
}).reset_index()

audit_f2['weighted_ctr_pct'] = (audit_f2['clicks_90d'] / audit_f2['impressions_90d']) * 100
display(audit_f2.sort_values('tier'))

--- Auditing Finding #1: Intent-Based Traffic Value ---


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,main_intent,PAGES,CLICKS,CPC,VALUE
3,transactional,55488,1712535.0,0.76,1301526.60
4,commercial,50809,1070013.0,0.68,727608.84
2,informational,242222,3574548.0,0.15,536182.20
1,navigational,1911,9497.0,0.35,3323.95
0,None,76862,74244.0,0.00,0.00



--- Auditing Finding #2: Weighted CTR by Position Tier ---


,tier,clicks_90d,impressions_90d,weighted_ctr_pct
0,1. Top 3,147951,31615691,0.467967
1,2. Page 1 (4-10),273863,132134544,0.207261
2,3. Striking,25712,19669342,0.130721
3,4. Page 3-5,10481,15849881,0.066127
4,5. Deep,1911,12473558,0.015320


**Finding #1: Captured Traffic Value**<br>
**Label Source:** The label 'Value' is a proxy, not a direct measurement. It is derived from Clicks * CPC Benchmark.<br>
**Validation Design Issue:** Your audit shows informational intent has 3.5M clicks, while the paper claimed 486.9K. This suggests the paper used a subset or a specific date range, while the warehouse contains the full history. The design relies on the assumption that external CPC benchmarks apply equally to all internal traffic, which might inflate value if the site's niche has lower actual costs.<br><br>
**Finding #2: Click Capture by Position Tier**<br>
**Label Source:** The label is a Weighted CTR (Sum(Clicks) / Sum(Impressions)).<br>
**Validation Design Issue:** Your calculated Top 3 CTR is 0.467%, while the paper claimed 0.423%. Because these numbers are so close, the design 'carries the claim' (it is reproducible). However, notice how the CTR drops significantly after Page 1. The validation design is strong because it uses a portfolio-wide average, but it could be 'leaky' if a few high-authority clients are skewing the average for the whole dataset.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.